In [0]:
import dlt
from pyspark.sql.functions import col, to_timestamp, when, lit, coalesce, regexp_replace, split, broadcast
from pyspark.sql.types import LongType

def ip_dotted_to_long(ip_col):
    """Convert IPv4 dotted-quad string to long using native Spark expressions."""
    parts = split(ip_col, "\\.")
    return (
        parts[0].cast("long") * 16777216 +
        parts[1].cast("long") * 65536 +
        parts[2].cast("long") * 256 +
        parts[3].cast("long")
    )

In [0]:
@dlt.view(
    name="electroniz_bronze_logs_geo_view",
    comment="Streaming view joining website logs with geolocation via IP range lookup"
)
def electroniz_bronze_logs_geo_view():
    logs = spark.readStream.table(
        "electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_website_logs"
    )
    logs_parsed = logs.withColumn(
        "time_clean",
        regexp_replace(col("time"), r"^\[|(\s[+-]\d{4})?\]$", "")
    ).withColumn(
        "event_time",
        coalesce(
            to_timestamp(col("time_clean"), "dd/MM/yyyy:HH:mm:ss"),
            to_timestamp(col("time_clean"), "dd/MMM/yyyy:HH:mm:ss")
        )
    ).withColumn(
        "remote_ip_long", ip_dotted_to_long(col("remote_ip"))
    )

    geo = spark.read.table(
        "electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_geolocation"
    )
    geo_parsed = geo.withColumn(
        "ip1_long", col("ip1").cast("long")
    ).withColumn(
        "ip2_long", col("ip2").cast("long")
    )

    joined = logs_parsed.join(
        broadcast(geo_parsed),
        (logs_parsed["remote_ip_long"] >= geo_parsed["ip1_long"]) &
        (logs_parsed["remote_ip_long"] <= geo_parsed["ip2_long"]),
        "left"
    )

    return joined.select(
        logs_parsed["agent"],
        logs_parsed["remote_ip"],
        logs_parsed["request"],
        logs_parsed["response"],
        logs_parsed["event_time"],
        coalesce(geo_parsed["country_code"], lit("Unknown")).alias("country_code"),
        coalesce(geo_parsed["country_name"], lit("Unknown")).alias("country_name"),
        when(col("agent").rlike("(?i)Edg(e|/)"), lit("Edge"))
        .when(col("agent").rlike("(?i)OPR|Opera"), lit("Opera"))
        .when(col("agent").rlike("(?i)Firefox"), lit("Firefox"))
        .when(col("agent").rlike("(?i)Chrome"), lit("Chrome"))
        .when(col("agent").rlike("(?i)MSIE|Trident"), lit("Internet Explorer"))
        .when(col("agent").rlike("(?i)Safari"), lit("Safari"))
        .otherwise(lit("Other")).alias("browser_type")
    )

In [0]:
@dlt.table(
    name="electroniz_silver_logs_geolocation",
    comment="Silver logs enriched with geolocation data",
    table_properties={"quality": "silver"}
)
def electroniz_silver_logs_geolocation():
    return spark.readStream.table("electroniz_bronze_logs_geo_view")